# 05season: data for the whole 2026 haze season

Part B of PF5 (the season and the indirect effect). This notebook downloads and prepares; the analysis comes in `06season`.

| Section | Source | What | Time |
|---|---|---|---|
| 2 | GEOS-FP (NASA, OPeNDAP) | smoke, PM2.5, sea salt, aerosol optical depth (optional: smoke column and flux), then daily means | depends on the server: 20 min to over an hour |
| 3 | NASA FIRMS | VIIRS fire hotspots, counted per day in 0.5 degree cells | about 10 min |
| 4 | IBTrACS (already downloaded in 01event) | all western Pacific tropical storms of the season | seconds |
| 5 | ERA5 daily statistics (downloaded by Likren from the Climate Data Store) | wind, rain, pressure, boundary layer height, merged into one file | seconds |

**Keeping the downloads small.** For a season of about 90 days, GEOS-FP is fetched at every second grid point (0.5 x 0.625 degrees, the same grid as MERRA-2, which the climatology in Part C will use), in 5-day pieces. (The first run also skipped every second time step; those files are kept, and four samples a day are enough for a daily mean.) FIRMS hotspots are counted per cell and day straight away, instead of keeping millions of rows.

Each step skips files that already exist, so the notebook can be re-run after an interruption, and re-run later to add the last days of October.

## Section 0: settings

In [2]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
KEYS_FILE = PROJECT_DIR / "config" / "keys.toml"
RAW_DIR = PROJECT_DIR / "data" / "raw" / "season"
EVENT_DIR = PROJECT_DIR / "data" / "raw" / "event"
OUT_DIR = PROJECT_DIR / "data" / "processed" / "season"

SEASON_START = "2026-08-01"  # <- first day of the season
SEASON_END = "2026-10-25"  # <- last day (ERA5 is available up to here for now; raise it later for the top-up)

BOX_WEST = 90.0  # <-
BOX_EAST = 160.0  # <-
BOX_SOUTH = -15.0  # <-
BOX_NORTH = 45.0  # <-

GEOSFP_SPACE_STEP = 2  # <- 2 = every second grid point (0.5 x 0.625 deg); 1 = full 0.25 x 0.3125 deg
GEOSFP_TIME_STEP = 1  # <- 1 = all 8 steps per day (contiguous requests are kinder to the server); 2 = every second step
GEOSFP_VARS = ["pm25", "ocsmass", "brsmass", "bcsmass", "sssmass25", "totexttau"]  # <- needed for 06season
GEOSFP_EXTRA = ["occmass", "brcmass", "bccmass", "ocfluxu", "ocfluxv", "bcfluxu", "bcfluxv", "brfluxu", "brfluxv"]  # <- smoke column and flux (optional)
DO_GEOSFP_EXTRA = False  # <- True to also fetch the optional columns and fluxes (slow server: leave for an overnight run)

FIRMS_SOURCES = ["VIIRS_SNPP_NRT", "VIIRS_NOAA20_NRT"]  # <- two satellites keep the counts steady if one has a gap
FIRMS_CELL_DEG = 0.5  # <- hotspot counting cells

DO_GEOSFP = True  # <-
DO_FIRMS = True  # <-
DO_TRACKS = True  # <-
DO_ERA5 = True  # <-

## Section 1: imports, folders, keys and helpers

In [3]:
import io
import time
import tomllib
import zipfile
import warnings
import requests
import numpy as np
import pandas as pd
import xarray as xr

warnings.filterwarnings("ignore", message="Ambiguous reference date string")
RAW_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
with open(KEYS_FILE, "rb") as f:
    KEYS = tomllib.load(f)

DAYS = pd.date_range(SEASON_START, SEASON_END, freq="D")
MONTHS = sorted(set(DAYS.strftime("%Y-%m")))
RESULTS = {}
print("Season:", DAYS[0].date(), "to", DAYS[-1].date(), f"({len(DAYS)} days), months {MONTHS}")


def done(path):
    return path.exists() and path.stat().st_size > 0


def open_geosfp(url):
    for engine in ["netcdf4", "pydap"]:
        try:
            target = url.replace("https://", "dap2://") if engine == "pydap" else url
            kwargs = {"timeout": GEOSFP_TIMEOUT} if engine == "pydap" else {}
            ds = xr.open_dataset(target, engine=engine, **kwargs)
            print("GEOS-FP opened with", engine)
            return ds
        except Exception as e:
            print(f"GEOS-FP engine {engine} failed:", type(e).__name__)
    raise OSError("no engine could open GEOS-FP")

Season: 2026-08-01 to 2026-10-25 (86 days), months ['2026-08', '2026-09', '2026-10']


## Section 2: GEOS-FP for the season

The NASA server often drops long answers ("ChunkedEncodingError"), so requests are kept small: one variable, 5 days at a time, saved as pieces (`geosfp_pm25_2026-09_part01.nc`, ...). Finished pieces are never fetched again, so after a bad run just run the notebook again and only the missing pieces are tried. Files from the first run (whole months, for example `geosfp_pm25_2026-08.nc`) are kept and used as they are.

Then daily means of everything downloaded so far go to `data/processed/season/geosfp_daily.nc`, with a count of days per variable so gaps are visible.

The six variables needed for `06season` come first. The optional smoke column and flux variables (`DO_GEOSFP_EXTRA` in Section 0) are best left for an overnight run.

In [4]:
GEOSFP_URL = "https://opendap.nccs.nasa.gov/dods/GEOS-5/fp/0.25_deg/assim/tavg3_2d_aer_Nx"
GEOSFP_TRIES = 5  # <- attempts per piece
GEOSFP_WAIT = 30  # <- seconds to wait before a retry
GEOSFP_TIMEOUT = 240  # <- seconds before a stuck request is given up
CHUNK_DAYS = 5  # <- days per request: small pieces fail less often and a failure costs less

if DO_GEOSFP:
    try:
        geos = open_geosfp(GEOSFP_URL)
        gt = pd.to_datetime([str(t) for t in geos.time.values])
        lat = geos.lat.values
        lon = geos.lon.values
        j0 = int(np.abs(lat - BOX_SOUTH).argmin())
        j1 = int(np.abs(lat - BOX_NORTH).argmin())
        i0 = int(np.abs(lon - BOX_WEST).argmin())
        i1 = int(np.abs(lon - BOX_EAST).argmin())
        wanted = GEOSFP_VARS + (GEOSFP_EXTRA if DO_GEOSFP_EXTRA else [])
        failed = []
        for v in wanted:
            for month in MONTHS:
                if done(RAW_DIR / f"geosfp_{v}_{month}.nc"):
                    continue
                m_days = DAYS[DAYS.strftime("%Y-%m") == month]
                chunks = [m_days[k:k + CHUNK_DAYS] for k in range(0, len(m_days), CHUNK_DAYS)]
                for k, cd in enumerate(chunks, start=1):
                    piece = RAW_DIR / f"geosfp_{v}_{month}_part{k:02d}.nc"
                    if done(piece):
                        continue
                    idx = np.where((gt >= cd[0]) & (gt < cd[-1] + pd.Timedelta(days=1)))[0]
                    if len(idx) == 0:
                        continue
                    t_sel = slice(int(idx[0]), int(idx[-1]) + 1, GEOSFP_TIME_STEP)
                    for attempt in range(1, GEOSFP_TRIES + 1):
                        try:
                            t0 = time.time()
                            part = geos[v].isel(time=t_sel, lat=slice(j0, j1 + 1, GEOSFP_SPACE_STEP), lon=slice(i0, i1 + 1, GEOSFP_SPACE_STEP)).load()
                            part.astype("float32").to_netcdf(piece)
                            print(f"{v} {cd[0]:%d %b} to {cd[-1]:%d %b}: {part.shape} in {time.time() - t0:.0f} s")
                            break
                        except Exception as e:
                            print(f"{v} {cd[0]:%d %b} to {cd[-1]:%d %b}: attempt {attempt} failed after {time.time() - t0:.0f} s:", type(e).__name__)
                            time.sleep(GEOSFP_WAIT)
                    else:
                        failed.append(f"{v} {cd[0]:%d %b}")
        RESULTS["GEOS-FP download"] = "OK" if not failed else f"PARTLY ({len(failed)} pieces failed: {', '.join(failed)}); re-run to retry only these"
    except Exception as e:
        print("GEOS-FP step failed:", type(e).__name__, e)
        RESULTS["GEOS-FP download"] = f"FAIL ({type(e).__name__})"

GEOS-FP engine netcdf4 failed: OSError


.\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GEOS-FP opened with pydap
pm25 01 Sep to 05 Sep: (40, 121, 113) in 17 s
pm25 06 Sep to 10 Sep: (40, 121, 113) in 17 s
pm25 11 Sep to 15 Sep: (40, 121, 113) in 17 s
pm25 16 Sep to 20 Sep: (40, 121, 113) in 20 s
pm25 21 Sep to 25 Sep: (40, 121, 113) in 19 s
pm25 26 Sep to 30 Sep: (40, 121, 113) in 20 s
pm25 01 Oct to 05 Oct: (40, 121, 113) in 23 s
pm25 06 Oct to 10 Oct: (40, 121, 113) in 23 s
pm25 11 Oct to 15 Oct: (40, 121, 113) in 22 s
pm25 16 Oct to 20 Oct: (40, 121, 113) in 20 s
pm25 21 Oct to 25 Oct: (40, 121, 113) in 20 s
ocsmass 01 Sep to 05 Sep: (40, 121, 113) in 18 s
ocsmass 06 Sep to 10 Sep: (40, 121, 113) in 18 s
ocsmass 11 Sep to 15 Sep: (40, 121, 113) in 17 s
ocsmass 16 Sep to 20 Sep: (40, 121, 113) in 18 s
ocsmass 21 Sep to 25 Sep: (40, 121, 113) in 17 s
ocsmass 26 Sep to 30 Sep: (40, 121, 113) in 17 s
brsmass 01 Aug to 05 Aug: (40, 121, 113) in 17 s
brsmass 06 Aug to 10 Aug: (40, 121, 113) in 23 s
brsmass 11 Aug to 15 Aug: (40, 121, 113) in 20 s
brsmass 16 Aug to 20 Aug: (

In [10]:
# Daily means of everything downloaded so far
daily_file = OUT_DIR / "geosfp_daily.nc"
try:
    parts = []
    for v in GEOSFP_VARS + GEOSFP_EXTRA:
        files = sorted(RAW_DIR.glob(f"geosfp_{v}_*.nc"))
        if not files:
            continue
        da = xr.concat([xr.open_dataset(f)[v] for f in files], dim="time").sortby("time")
        da = da.drop_duplicates("time")
        day_mean = da.resample(time="1D").mean().rename(v)
        parts.append(day_mean)
    full_days = pd.DatetimeIndex(DAYS)
    daily = xr.merge(parts, join="outer").reindex(time=full_days)
    print("Days with data per variable (of", len(full_days), "):")
    for v in daily.data_vars:
        n = int((~daily[v].isnull().all(["lat", "lon"])).sum())
        flag = "" if n == len(full_days) else "   <- incomplete"
        print(f"  {v:10s} {n}{flag}")
    for v in ["pm25", "ocsmass", "brsmass", "bcsmass", "sssmass25"]:
        if v in daily:
            daily[v] = daily[v] * 1e9
            daily[v].attrs["units"] = "micrograms per m3"
    if all(v in daily for v in ["ocsmass", "brsmass", "bcsmass"]):
        daily["smoke"] = daily["ocsmass"] + daily["brsmass"] + daily["bcsmass"]
        daily["smoke"].attrs["units"] = "micrograms per m3"
    if all(v in daily for v in ["occmass", "brcmass", "bccmass"]):
        daily["smoke_col"] = (daily["occmass"] + daily["brcmass"] + daily["bccmass"]) * 1e6
        daily["smoke_col"].attrs["units"] = "mg per m2"
    if all(v in daily for v in ["ocfluxu", "bcfluxu", "brfluxu"]):
        daily["smoke_fu"] = daily["ocfluxu"] + daily["bcfluxu"] + daily["brfluxu"]
        daily["smoke_fv"] = daily["ocfluxv"] + daily["bcfluxv"] + daily["brfluxv"]
    daily.to_netcdf(daily_file)
    print("Daily GEOS-FP:", dict(daily.sizes), "variables:", list(daily.data_vars))
    print("First and last day:", str(daily.time.values[0])[:10], str(daily.time.values[-1])[:10])
    RESULTS["GEOS-FP daily means"] = "OK"
except Exception as e:
    print("Daily means failed:", type(e).__name__, e)
    RESULTS["GEOS-FP daily means"] = f"FAIL ({type(e).__name__})"

Days with data per variable (of 86 ):
  pm25       86
  ocsmass    86
  brsmass    86
  bcsmass    86
  sssmass25  86
  totexttau  86
  occmass    56   <- incomplete
  brcmass    56   <- incomplete
  bccmass    30   <- incomplete
  ocfluxv    55   <- incomplete
  bcfluxu    55   <- incomplete
  bcfluxv    55   <- incomplete
  brfluxu    56   <- incomplete
  brfluxv    25   <- incomplete
Daily GEOS-FP: {'lat': 121, 'lon': 113, 'time': 86} variables: ['pm25', 'ocsmass', 'brsmass', 'bcsmass', 'sssmass25', 'totexttau', 'occmass', 'brcmass', 'bccmass', 'ocfluxv', 'bcfluxu', 'bcfluxv', 'brfluxu', 'brfluxv', 'smoke', 'smoke_col']
First and last day: 2026-08-01 2026-10-25


## Section 3: FIRMS hotspots per day and 0.5 degree cell

For each satellite and day: number of hotspots and summed fire radiative power (FRP, megawatts; a measure of how intensely the fires burn) in every 0.5 degree cell. Low-confidence detections are left out. Only these counts are kept (`firms_cells_<source>.csv`), not the individual hotspots.

FIRMS keeps "near real time" (NRT) data until the reprocessed "standard" (SP) version replaces it a few months later. If a day returns nothing from NRT, the cell tries the SP version of the same satellite.

In [11]:
FIRMS_KEY = KEYS["firms"]["key"]
box_text = f"{BOX_WEST:g},{BOX_SOUTH:g},{BOX_EAST:g},{BOX_NORTH:g}"


def firms_day(source, day):
    url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{FIRMS_KEY}/{source}/{box_text}/1/{day.date()}"
    r = requests.get(url, timeout=180)
    if r.status_code != 200:
        return None, f"HTTP {r.status_code}"
    df = pd.read_csv(io.StringIO(r.text))
    if "latitude" not in df.columns:
        return None, r.text[:100].replace(FIRMS_KEY, "***")
    return df, "ok"


if DO_FIRMS:
    try:
        for source in FIRMS_SOURCES:
            out_file = RAW_DIR / f"firms_cells_{source}.csv"
            have = pd.read_csv(out_file, parse_dates=["date"]) if done(out_file) else pd.DataFrame(columns=["date"])
            have_days = set(pd.to_datetime(have["date"]).dt.normalize()) if len(have) else set()
            new = []
            counts = []
            for day in DAYS:
                if day in have_days:
                    continue
                df, msg = firms_day(source, day)
                if (df is None or len(df) == 0) and source.endswith("_NRT"):
                    df, msg = firms_day(source.replace("_NRT", "_SP"), day)
                if df is None:
                    print(f"{source} {day.date()}: {msg}")
                    continue
                df = df[df["confidence"].astype(str) != "l"]
                cl = (np.floor(df["latitude"] / FIRMS_CELL_DEG) + 0.5) * FIRMS_CELL_DEG
                cn = (np.floor(df["longitude"] / FIRMS_CELL_DEG) + 0.5) * FIRMS_CELL_DEG
                agg = df.assign(cell_lat=cl, cell_lon=cn).groupby(["cell_lat", "cell_lon"]).agg(hotspots=("frp", "size"), frp_sum=("frp", "sum")).reset_index()
                agg.insert(0, "date", day)
                new.append(agg)
                counts.append(len(df))
            if new:
                allc = pd.concat([have] + new, ignore_index=True) if len(have) else pd.concat(new, ignore_index=True)
                allc["date"] = pd.to_datetime(allc["date"])
                allc = allc.sort_values(["date", "cell_lat", "cell_lon"])
                allc.to_csv(out_file, index=False)
                print(f"{source}: {len(new)} new days, {sum(counts):,} hotspots, file now {len(allc):,} rows")
            else:
                print(f"{source}: nothing new")
        RESULTS["FIRMS"] = "OK"
    except Exception as e:
        print("FIRMS step failed:", type(e).__name__, str(e).replace(FIRMS_KEY, "***"))
        RESULTS["FIRMS"] = f"FAIL ({type(e).__name__})"

VIIRS_SNPP_NRT: 1 new days, 0 hotspots, file now 77,052 rows
VIIRS_NOAA20_NRT: nothing new


## Section 4: western Pacific storms of the season (IBTrACS)

From the IBTrACS file downloaded in `01event`. Keeps every western Pacific (basin WP) system with at least one track point in the season. Re-download the IBTrACS file later (delete it in `data/raw/event/` and re-run Section 7 of `01event`) to pick up the final October tracks.

In [12]:
tracks_file = OUT_DIR / "season_tracks.csv"
if DO_TRACKS:
    try:
        ib = pd.read_csv(EVENT_DIR / "ibtracs_last3years.csv", skiprows=[1], low_memory=False)
        ib["time"] = pd.to_datetime(ib["ISO_TIME"])
        ib = ib[(ib["BASIN"] == "WP") & (ib["time"] >= DAYS[0]) & (ib["time"] < DAYS[-1] + pd.Timedelta(days=1))].copy()
        keep = [c for c in ["SID", "NAME", "SEASON", "ISO_TIME", "time", "NATURE", "LAT", "LON", "USA_WIND", "USA_PRES", "USA_SSHS", "TOKYO_WIND", "TOKYO_PRES"] if c in ib.columns]
        ib = ib[keep]
        ib.to_csv(tracks_file, index=False)
        storms = ib.groupby(["SID", "NAME"]).agg(first=("time", "min"), last=("time", "max"), max_wind_kt=("USA_WIND", lambda s: pd.to_numeric(s, errors="coerce").max())).sort_values("first")
        print(f"Western Pacific systems with track points between {DAYS[0].date()} and {DAYS[-1].date()}: {len(storms)}")
        print(storms.to_string())
        print("Latest track time in the file:", ib["time"].max())
        RESULTS["IBTrACS season"] = "OK"
    except Exception as e:
        print("Track step failed:", type(e).__name__, e)
        RESULTS["IBTrACS season"] = f"FAIL ({type(e).__name__})"

Western Pacific systems with track points between 2026-08-01 and 2026-10-25: 22
                                      first                last  max_wind_kt
SID           NAME                                                          
2026208N13178 DOLPHIN   2026-08-01 00:00:00 2026-08-09 18:00:00        124.0
2026217N18119 PEILOU    2026-08-05 00:00:00 2026-08-13 18:00:00         39.0
2026219N29161 CHAN-HOM  2026-08-07 00:00:00 2026-08-12 00:00:00         51.0
2026222N26160 UNNAMED   2026-08-10 00:00:00 2026-08-11 18:00:00         35.0
2026225N23145 NANGKA    2026-08-12 12:00:00 2026-08-16 06:00:00         35.0
2026231N08155 SAUDEL    2026-08-18 12:00:00 2026-08-28 00:00:00        115.0
2026233N20108 NARRA     2026-08-20 18:00:00 2026-08-26 06:00:00         39.0
2026235N12135 ATSANI    2026-08-22 12:00:00 2026-08-25 18:00:00         29.0
2026238N17161 BANG-LANG 2026-08-26 06:00:00 2026-08-31 00:00:00         74.0
2026239N20168 ETAU      2026-08-27 00:00:00 2026-08-31 00:00:00         4

## Section 5: ERA5 daily statistics (from the Climate Data Store website)

Reads every `era5_daily_single_*.zip` in `data/raw/season/`, joins the months, and writes `data/processed/season/era5_daily.nc` with:

- `u10`, `v10`: daily mean 10 m wind (m/s)
- `msl`: daily mean sea level pressure, converted to hPa
- `tp`: daily rainfall, converted to mm per day (the daily mean of hourly totals x 24 x 1000)
- `blh`: daily mean boundary layer height (m), the depth of the well-mixed air near the ground; a low value traps smoke near the surface

To add the last days of October later: download them from the website with the same settings (month 10, days 26 to 31), save the zip in `data/raw/season/` with a new name (for example `era5_daily_single_2026-10b.zip`), and re-run this section.

In [13]:
era5_file = OUT_DIR / "era5_daily.nc"
if DO_ERA5:
    try:
        zips = sorted(RAW_DIR.glob("era5_daily_single_*.zip"))
        print("ERA5 zips:", [z.name for z in zips])
        months = []
        tmp_dir = RAW_DIR / "_unzipped"
        tmp_dir.mkdir(exist_ok=True)
        for z in zips:
            with zipfile.ZipFile(z) as zf:
                ds_vars = []
                for name in zf.namelist():
                    path = Path(zf.extract(name, tmp_dir))
                    with xr.open_dataset(path) as ds:
                        ds_vars.append(ds.load())
                    path.unlink()
                months.append(xr.merge(ds_vars, compat="override"))
        tmp_dir.rmdir()
        era5 = xr.concat(months, dim="valid_time").sortby("valid_time")
        era5 = era5.drop_duplicates("valid_time").rename({"valid_time": "time", "latitude": "lat", "longitude": "lon"}).sortby("lat")
        era5 = era5[["u10", "v10", "msl", "tp", "blh"]]
        era5["msl"] = era5["msl"] / 100
        era5["msl"].attrs["units"] = "hPa"
        era5["tp"] = era5["tp"] * 24 * 1000
        era5["tp"].attrs["units"] = "mm per day"
        era5.to_netcdf(era5_file)
        t = pd.to_datetime(era5.time.values)
        gaps = pd.date_range(t[0], t[-1], freq="D").difference(t)
        print("ERA5 daily:", dict(era5.sizes), " from", t[0].date(), "to", t[-1].date(), " missing days:", len(gaps))
        print("Mean rainfall over the box (mm/day):", round(float(era5["tp"].mean()), 1), " boundary layer height range (m):", round(float(era5["blh"].min())), "to", round(float(era5["blh"].max())))
        RESULTS["ERA5 daily"] = "OK"
    except Exception as e:
        print("ERA5 step failed:", type(e).__name__, e)
        RESULTS["ERA5 daily"] = f"FAIL ({type(e).__name__})"

ERA5 zips: ['era5_daily_single_2026-08-09.zip', 'era5_daily_single_2026-10.zip']
ERA5 daily: {'time': 86, 'lat': 241, 'lon': 281}  from 2026-08-01 to 2026-10-25  missing days: 0
Mean rainfall over the box (mm/day): 4.7  boundary layer height range (m): 21 to 2263


## Section 6: summary

In [14]:
print(pd.DataFrame(list(RESULTS.items()), columns=["Step", "Result"]).to_string(index=False))
print()
for folder in [RAW_DIR, OUT_DIR]:
    files = sorted(p for p in folder.glob("*") if p.is_file())
    print(f"{folder.relative_to(PROJECT_DIR)}: {len(files)} files, {sum(p.stat().st_size for p in files) / 1e6:.0f} MB")

               Step Result
   GEOS-FP download     OK
GEOS-FP daily means     OK
              FIRMS     OK
     IBTrACS season     OK
         ERA5 daily     OK

data\raw\season: 83 files, 333 MB
data\processed\season: 3 files, 168 MB


___
**END**